<a href="https://colab.research.google.com/github/Mansi-3s/Creditt_wise_loan/blob/main/Image_Augmentation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Task


Image Augmentation Implementation in AI.

You are data scientist working on a medical image classification project to detect skin cancer from dermoscopy images. Your dataset contains only 1,000 labeled images (500 benign, 500 malignant), which is insufficient for training a robust deep learning model.

Part A: Image Augmentation Strategy
Design a comprehensive image augmentation pipeline for this medical imaging task. Your answer should include:

1. Geometric Transformations: List 3-4 geometrric augmentations appropriate for medical images and explain why each is suitable.
1. Color/Intensity Augmentations: Describe 2-3 colour/intensity transformations that preserve diagnostic information while creating variations.
3. Advanced Techniques: Explain how you would implement either Mixup OR Cutmix for this specific medical imaging task. What are the potential benefits and risks.
4. Implementation Approach: Outline how you would implement this pipeline using the albumentations library (or similar). Include cinsiderations for train/validation split.

In [39]:

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [40]:
data_dir = "dataset"

data = []

for label_name in ["benign", "malignant"]:
    folder = os.path.join(data_dir, label_name)

    for img in os.listdir(folder):
        data.append({
            "image_path": os.path.join(folder, img),
            "label": 0 if label_name == "benign" else 1
        })

df = pd.DataFrame(data)

print("Total images:", len(df))

Total images: 10


In [41]:
import os
import cv2
import numpy as np

# Define the base directory for the dataset
data_dir = "dataset"

# Create the main dataset directory if it doesn't exist
os.makedirs(data_dir, exist_ok=True)

# Define the subdirectories for benign and malignant images
benign_dir = os.path.join(data_dir, "benign")
malignant_dir = os.path.join(data_dir, "malignant")

# Create the subdirectories if they don't exist
os.makedirs(benign_dir, exist_ok=True)
os.makedirs(malignant_dir, exist_ok=True)

print(f"Created directory: {benign_dir}")
print(f"Created directory: {malignant_dir}")

# Create some dummy image files (e.g., empty files) for demonstration
num_dummy_images = 5

# Generate actual dummy image files
for i in range(num_dummy_images):
    dummy_image = np.zeros((224, 224, 3), dtype=np.uint8) # Create a black image

    # Save dummy benign image
    cv2.imwrite(os.path.join(benign_dir, f"benign_image_{i}.jpg"), dummy_image)

    # Save dummy malignant image
    cv2.imwrite(os.path.join(malignant_dir, f"malignant_image_{i}.jpg"), dummy_image)

print(f"Created {num_dummy_images} dummy images in each folder.")

Created directory: dataset/benign
Created directory: dataset/malignant
Created 5 dummy images in each folder.


In [42]:
import pandas as pd

train_df, val_df = train_test_split(df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [43]:
train_transforms = A.Compose([
    A.Resize(224,224),

    # Geometric (safe for dermoscopy)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.5
    ),

    # Color/Intensity (controlled)
    A.RandomBrightnessContrast(0.15,0.15,p=0.5),
    A.HueSaturationValue(5,10,10,p=0.3),
    A.GaussNoise(p=0.2),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [44]:
val_transforms = A.Compose([
    A.Resize(224,224),
    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),
    ToTensorV2()
])

In [45]:
class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.loc[idx, "image_path"]
        label = self.df.loc[idx, "label"]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, torch.tensor(label, dtype=torch.float32)

In [46]:
train_dataset = SkinCancerDataset(train_df, train_transforms)
val_dataset = SkinCancerDataset(val_df, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [47]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [48]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size)

    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [50]:
for epoch in range(10):

    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images, y_a, y_b, lam = mixup_data(images, labels)

        outputs = model(images).squeeze()

        loss = lam*criterion(outputs, y_a) + \
               (1-lam)*criterion(outputs, y_b)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed
Epoch 3 completed
Epoch 4 completed
Epoch 5 completed
Epoch 6 completed
Epoch 7 completed
Epoch 8 completed
Epoch 9 completed
Epoch 10 completed


In [51]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images).squeeze()
        preds = torch.sigmoid(outputs) > 0.5

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", correct/total)

Validation Accuracy: 0.5
